# 01 — Prepare data and cache the teacher (run ONCE, about 1.5–3 h)

**Notebook settings (right panel):**
1. *Accelerator*: **GPU T4 x2** (one GPU is used here).
2. *Internet*: **On** (needs a phone-verified Kaggle account).
3. *Add Input* → *Datasets*: add one copy of **ISIC 2019** (the training images, ~25k `ISIC_*.jpg`)
   and one copy of **PAD-UFES-20** (has `metadata.csv` with a `diagnostic` column and `PAT_*.png` images).

Then click **Save Version → Save & Run All (Commit)**. You can close the browser; it runs in the background.
When it finishes, its *Output* is the input for notebook 02.

In [ ]:
!nvidia-smi -L
%cd /kaggle/working
!rm -rf IEEE-TMI && git clone -q https://github.com/Nafi14078/IEEE-TMI.git
%cd /kaggle/working/IEEE-TMI
!pip install -q open_clip_torch
!pip install -q -e . --no-deps

In [ ]:
# Locate the datasets you added as inputs. Kaggle mounts them at different depths
# (e.g. /kaggle/input/datasets/<owner>/<name>/...), sometimes through links, so search everything.
import os, urllib.request
from collections import Counter
import pandas as pd

IMG = ('.jpg', '.jpeg', '.png')
isic_n, pad_n, metas, zips, seen = Counter(), Counter(), [], [], set()
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    real = os.path.realpath(root)
    if real in seen:
        dirs[:] = []
        continue
    seen.add(real)
    ni = sum(f.startswith('ISIC_') and f.lower().endswith(IMG) for f in files)
    npd = sum(f.startswith('PAT_') and f.lower().endswith(IMG) for f in files)
    path = root
    while path.startswith('/kaggle/input'):          # add counts to every ancestor folder
        isic_n[path] += ni
        pad_n[path] += npd
        path = os.path.dirname(path)
    if 'metadata.csv' in files:
        metas.append(os.path.join(root, 'metadata.csv'))
    zips += [os.path.join(root, f) for f in files if f.lower().endswith('.zip')]

def deepest_holding(counter, frac=0.98):
    # deepest folder that contains (almost) all of the matching images
    total = counter['/kaggle/input']
    cands = [p for p, n in counter.items() if total and n >= frac * total]
    return max(cands, key=lambda p: p.count('/')) if cands else None

print('Top-level input folders:')
for p, n in sorted(isic_n.items()):
    if p.count('/') <= 5:
        print(f'  {p:70s} ISIC imgs: {n:6d}   PAT imgs: {pad_n[p]:5d}')
if zips:
    print('Zip files found (images inside zips are NOT used):', zips[:5])

isic_root = deepest_holding(isic_n)
print()
print('ISIC images:', isic_n['/kaggle/input'], 'under', isic_root)
assert isic_n['/kaggle/input'] > 20000, (
    'Fewer than 20 000 ISIC_*.jpg images found. Add an ISIC 2019 TRAINING-set dataset '
    '(~25 331 images, unzipped) with "+ Add Input". See the folder list printed above.')

pad_meta = next((m for m in metas if {'diagnostic', 'img_id'} <= set(pd.read_csv(m, nrows=1).columns)), None)
assert pad_meta, 'No PAD-UFES-20 metadata.csv (with a "diagnostic" column) found. Add a PAD-UFES-20 dataset.'
pad_root = os.path.dirname(pad_meta)
while pad_n[pad_root] < 2000 and pad_root != '/kaggle/input':   # go up until the images are inside
    pad_root = os.path.dirname(pad_root)
print('PAD-UFES-20:', pad_meta, '| images:', pad_n[pad_root], 'under', pad_root)
assert pad_n[pad_root] >= 2000, 'PAD-UFES-20 images (PAT_*.png) not found next to metadata.csv'

# Official ISIC 2019 label + metadata CSVs (small), so lesion_id grouping is always available.
os.makedirs('/kaggle/tmp_raw', exist_ok=True)
base = 'https://isic-challenge-data.s3.amazonaws.com/2019/'
for f in ['ISIC_2019_Training_GroundTruth.csv', 'ISIC_2019_Training_Metadata.csv']:
    dst = f'/kaggle/tmp_raw/{f}'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(base + f, dst)
print('ISIC CSVs downloaded')

In [ ]:
# Task 3: freeze splits (lesion-grouped 70/10/20, nested 1/5/10 % labels x 3 seeds)
!python scripts/prepare_splits.py --isic-gt /kaggle/tmp_raw/ISIC_2019_Training_GroundTruth.csv --isic-meta /kaggle/tmp_raw/ISIC_2019_Training_Metadata.csv --isic-images "{isic_root}" --pad-meta "{pad_meta}" --pad-images "{pad_root}"

In [ ]:
# Resize once (short side 256 px)
!python scripts/preprocess_images.py --workers 4
!du -sh data/processed

In [ ]:
# Task 9: cache BiomedCLIP outputs (8 views per image + 8 prompt sets)
!python scripts/cache_teacher.py --teacher hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224 --out cache/biomedclip.pt --views 8 --batch 64 --workers 4

In [ ]:
# Task 4: does the label-free reliability predict when the teacher is right?  (GO / NO-GO)
for f in (1, 5, 10):
    !python scripts/check_reliability.py --set data.label_fraction={f} data.num_workers=4

In [ ]:
# Teacher baselines (zero-shot + linear probe) for every label fraction and seed
!python scripts/run_grid.py teacher
!rm -rf /kaggle/tmp_raw
!du -sh data cache runs results